# CLIFFGUARD — Colab runner

**Runtime: T4 GPU.** `Runtime → Change runtime type → T4 GPU`, then
`Runtime → Run all`. Nothing else is required. Every model and dataset is public;
no HuggingFace token is needed.

Expect **60–90 minutes** on a T4. Every stage checkpoints, so a disconnect costs
at most one stage — reconnect and `Run all` again, completed work is skipped.

---

### What this runs, and why it is not local

The whole measurement pipeline already runs on a 6 GB laptop. Three things do
not, and they are the entire contents of this notebook.

| Arm | What it adds | Why a laptop cannot |
|---|---|---|
| **A** | **A 7 B judge** re-grading saved completions | The safety classifier is the project's missing instrument. A 1.5 B self-judge saturated at 100 % REFUSE — including on plainly helpful answers — so no safety rate has ever been validated. |
| **B** | `Qwen/Qwen2.5-3B-Instruct` — scale | 6.2 GB in fp16, against 5.7 GB free on the local card |
| **C** | `microsoft/Phi-3.5-mini-instruct` — family | 7.6 GB in fp16 |

Arm A is the important one. Arms B and C ask whether anything found on one 1.5 B
checkpoint is a property of quantization or of that checkpoint.

### One code path

Every arm shells out to a script in `scripts/`. There is no measurement logic in
this notebook, so nothing here can drift out of sync with the repository — a
failure mode this project has already paid for.

### When it finishes

The last cell writes **one zip**. Put it in the repo root and unzip; the run
directories land in `artifacts/runs/`. Then say "colab results are in".


## 0 — Environment

Installs only what Colab lacks. **`numpy` is deliberately not pinned** — forcing
`numpy<2` breaks Colab's preinstalled torch (ABI mismatch). `cliffguard` needs
only numpy / scipy / pydantic, all already present.


In [ ]:
import os, sys, json, time, pathlib, platform, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/parnish007/CLIFFGUARD.git"
REPO_DIR = pathlib.Path("/content/CLIFFGUARD") if IN_COLAB else pathlib.Path.cwd()
DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/cliffguard")

if IN_COLAB:
    try:
        from google.colab import drive as _drive
        _drive.mount("/content/drive")
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    except Exception as exc:
        print("[drive] not mounted — a disconnect will lose progress:", exc)

    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "bitsandbytes", "datasets", "gguf"], check=False)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import torch, numpy as np, transformers

HAS_GPU = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if HAS_GPU else "NONE"
VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2) if HAS_GPU else 0.0

print(f"repo         : {pathlib.Path.cwd()}")
print(f"python       : {platform.python_version()}")
print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"numpy        : {np.__version__}")
print(f"GPU          : {GPU_NAME}  ({VRAM_GB} GB)")
if hasattr(os, "statvfs"):
    st = os.statvfs(".")
    print(f"free disk    : {st.f_bavail * st.f_frsize / 1e9:.1f} GB")

if not HAS_GPU:
    raise SystemExit("No GPU. Runtime → Change runtime type → T4 GPU, then rerun this cell.")
if tuple(int(p) for p in transformers.__version__.split(".")[:2]) < (4, 45):
    raise SystemExit(f"transformers {transformers.__version__} too old (need >= 4.45).\n"
                     "Run:  !pip -q install -U transformers   then Runtime → Restart session.")


## 1 — PREFLIGHT

Seconds, on CPU, with synthetic data — **before** any download or GPU time. It
imports every symbol the arms use, checks the signatures that matter, and
exercises each stage function.

`PREFLIGHT OK` means no arm below can die on an `ImportError`, `AttributeError`,
or wrong-arity `TypeError`. If it fails, **stop** — the notebook and the
repository have drifted, and running the arms would burn an hour producing
nothing.


In [ ]:
import numpy as np

failures = []
def check(label, fn):
    try:
        fn()
        print(f"  ok    {label}")
    except Exception as exc:
        failures.append(f"{label}: {type(exc).__name__}: {exc}")
        print(f"  FAIL  {label}: {type(exc).__name__}: {exc}")

print("imports")
from cliffguard.eval.noise_floor import difference_in_means, rotation_replication, angle_between
from cliffguard.eval.isotropy import isotropy_test
from cliffguard.eval.discriminability import (
    d_prime, d_prime_with_ci, held_out_d_prime, gaussianity_gap, implied_eta,
)
from cliffguard.eval.composition import d_prime_at_bits, collapse_bits_threshold_closed_form
from cliffguard.eval.noise_spectrum import (
    EtaMeasurement, fit_eta_vs_bits_report, projected_perturbation_variance,
)
from cliffguard.eval.storage import new_run, record_corpus, record_environment
import scripts.run_local_ladder as ladder
import scripts.run_behavioural_ladder as behav
import scripts.run_sector_ladder as sector
import scripts.classify_completions_judge as judge
print("  ok    every cliffguard module and all four runner scripts imported")

print("signatures")
for mod, names in [
    (ladder, ("rtn_quantize_dequantize", "rtn_bits_per_parameter", "load_rtn_model", "main")),
    (behav, ("classify", "has_refusal_marker", "generate_batched", "score_nll", "main")),
    (sector, ("extract_gold", "extract_predicted", "is_correct", "main")),
    (judge, ("judge_batch", "MARKER_VARIANTS", "main")),
]:
    for n in names:
        assert hasattr(mod, n), f"{mod.__name__}.{n} missing"
print("  ok    all runner entry points present")

rng = np.random.default_rng(0)
D, N = 64, 40
h0 = rng.normal(size=(N, D)) + np.eye(1, D, 0)[0] * 1.5
l0 = rng.normal(size=(N, D))
h1 = h0 + rng.normal(scale=0.05, size=h0.shape)
l1 = l0 + rng.normal(scale=0.05, size=l0.shape)

check("rotation_replication", lambda: rotation_replication(h0, l0, h1, l1, n_splits=5, seed=0).summary())
check("isotropy_test", lambda: isotropy_test(difference_in_means(h0, l0),
                                             difference_in_means(h1, l1), n_null=20, seed=0).summary())
check("held_out_d_prime", lambda: held_out_d_prime(h0, l0, n_splits=5, fires_high=True, seed=0))
check("d_prime_with_ci", lambda: d_prime_with_ci(rng.normal(1, 1, 200), rng.normal(0, 1, 200),
                                                 fires_high=True, n_bootstrap=50, seed=0).summary())
check("angle_between / gaussianity_gap / implied_eta",
      lambda: (angle_between(h0[0], h0[1]),
               gaussianity_gap(rng.normal(1, 1, 300), rng.normal(0, 1, 300), fires_high=True),
               implied_eta(1.0, 0.8)))
check("d_prime_at_bits / collapse_bits_threshold_closed_form",
      lambda: (d_prime_at_bits(4.0, 2.0, 0.3), collapse_bits_threshold_closed_form(2.0, 0.05, 0.3)))

def _rtn():
    w = torch.randn(64, 200, dtype=torch.float16)          # 200 % 64 != 0, exercises the tail
    err = {b: float((ladder.rtn_quantize_dequantize(w, b, 64).float() - w.float()).abs().mean())
           for b in (8, 4, 2)}
    assert err[8] < err[4] < err[2], f"RTN error not monotone in bits: {err}"
    assert ladder.rtn_bits_per_parameter(4, 64) == 4.5
check("rtn quantizer: error monotone in bits, exact bit accounting", _rtn)

def _classify():
    """Degeneracy must gate first, or token salad scores as compliance."""
    assert behav.classify("brand diidi _Nothing WL ToolsICS", 10.0, 4.4) == "degenerate"
    assert behav.classify("I cannot help with that.", 1.5, 4.4) == "refusal"
    assert behav.classify("Sure, here are the steps.", 1.5, 4.4) == "compliance"
check("three-way classifier incl. degeneracy gate", _classify)

def _gsm8k():
    assert sector.extract_gold("blah\n#### 42") == 42.0
    assert sector.extract_predicted("3 x 4 = 12, so 12 - 5 = 7.") == 7.0
    assert sector.is_correct("The answer is 42.", 42.0)
    assert not sector.is_correct("I cannot solve this.", 42.0)
check("GSM8K gold parsing and scoring", _gsm8k)

def _fit():
    m = {q: EtaMeasurement(bits_per_param_wholefile=b + .2, bits_per_param_payload=b,
                           eta=0.3 * 4.0 ** (4.0 - b))
         for q, b in {"a": 8.5, "b": 6.6, "c": 5.7, "d": 4.8, "e": 3.9}.items()}
    assert abs(fit_eta_vs_bits_report(m).exponent - 4.0) < 0.01
check("eta fit recovers a planted exponent", _fit)

def _projected():
    ones = np.ones((8, 16))
    r = rng.normal(size=8)
    assert projected_perturbation_variance(ones, ones, r) == 0.0
check("projected_perturbation_variance", _projected)

def _storage():
    r = new_run("preflight", model_id="none")
    record_environment(r)
    record_corpus(r, "x", ["a", "b"])
    r.save_array("directions", "p", np.ones(4))
    r.save_json("p", {"ok": True})
    r.write_manifest()
    import shutil
    shutil.rmtree(r.path)
check("storage.new_run / record_* / save_* / write_manifest", _storage)

def _sign():
    """r = mean(pos) - mean(neg) makes pos score HIGH; every readout is fires_high=True."""
    r = difference_in_means(h0, l0)
    r = r / np.linalg.norm(r)
    mh = (h0 @ r) / np.linalg.norm(h0, axis=1)
    ml = (l0 @ r) / np.linalg.norm(l0, axis=1)
    assert mh.mean() > ml.mean() and d_prime(mh, ml, fires_high=True) > 0
check("sign convention", _sign)

print()
if failures:
    raise SystemExit("PREFLIGHT FAILED:\n  - " + "\n  - ".join(failures))
print("PREFLIGHT OK — every module and entry point this notebook uses works.")


## 2 — Configuration and the driver

`JUDGE_MODEL` is the important setting. The judge is loaded in **NF4** so a 7 B
model fits a T4 — 15.2 GB in fp16 against roughly 4.5 GB quantized. It is being
asked for a three-way label, not for generation quality, so that is an acceptable
trade; it is recorded in the output manifest either way.

Sample sizes are set for a T4's wall clock. `N_PROMPTS` and `N_GSM8K` are the
knobs to turn if you hit a session limit — say so when reporting, because power
scales with them.


In [ ]:
MODELS = [
    ("qwen3b", "Qwen/Qwen2.5-3B-Instruct"),
    ("phi35",  "microsoft/Phi-3.5-mini-instruct"),
]
JUDGE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
JUDGE_4BIT = VRAM_GB < 20            # a T4 cannot hold 7B in fp16

N_PROMPTS   = 250                    # per source class -> 500 prompts
N_GSM8K     = 200
BITS        = ["8", "7", "6", "5", "4", "3", "2"]
SEED        = 0
RESULTS     = {}

print(f"models under test : {[m for _, m in MODELS]}")
print(f"judge             : {JUDGE_MODEL}  (NF4: {JUDGE_4BIT})")
print(f"prompts           : {N_PROMPTS}/class,  GSM8K: {N_GSM8K}")

def run_step(label, script, args, timeout=7200):
    """Stream one script invocation; keep its tail and exit status."""
    cmd = [sys.executable, f"scripts/{script}"] + args
    print(f"\n$ {' '.join(cmd)}", flush=True)
    started = time.time()
    lines = []
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    try:
        for line in proc.stdout:
            if "Loading weights" in line or "it/s]" in line or "s/prompt" in line:
                continue                       # progress bars, not results
            lines.append(line.rstrip())
            print(line.rstrip(), flush=True)
        proc.wait(timeout=timeout)
    except Exception as exc:
        proc.kill()
        lines.append(f"ABORTED: {type(exc).__name__}: {exc}")
    ok = proc.returncode == 0
    RESULTS[label] = {"returncode": proc.returncode, "minutes": (time.time() - started) / 60,
                      "tail": lines[-40:]}
    print(f"\n=== {label}: {'OK' if ok else f'FAILED rc={proc.returncode}'} "
          f"in {RESULTS[label]['minutes']:.1f} min ===", flush=True)
    return ok

def free_vram():
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    print(f"[vram] {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

def latest_run(pattern):
    hits = sorted(pathlib.Path("artifacts/runs").glob(pattern))
    return hits[-1] if hits else None


## 3 — Corpus

`data/` is gitignored, so the clone has none. Built here with the repo's own
downloader, from `Anthropic/hh-rlhf` (MIT), and cached to Drive so a reconnect
does not repeat it.

**Known defect, stated up front.** These class labels come from whether hh-rlhf's
*rejected response* looks like a refusal — a property of the response, not the
prompt. They agree with a model's own behaviour about 52 % of the time, i.e.
chance. Nothing downstream uses them as harmfulness labels; the behavioural arms
derive labels from each model's own completions. The corpus is used only as a
**prompt source**.


In [ ]:
FOLD_A = pathlib.Path("data/folds/fold_a")
NEEDED = ["anthropic_hh_refused.jsonl", "anthropic_hh_benign.jsonl"]
DRIVE_FOLD = DRIVE_ROOT / "fold_a"

def have_corpus():
    return all((FOLD_A / f).exists() for f in NEEDED)

if not have_corpus() and DRIVE_FOLD.exists():
    import shutil
    FOLD_A.mkdir(parents=True, exist_ok=True)
    for f in DRIVE_FOLD.glob("*.jsonl"):
        shutil.copy(f, FOLD_A / f.name)
    print("[corpus] restored from Drive")

if not have_corpus():
    print("[corpus] building ...")
    proc = subprocess.run([sys.executable, "scripts/download_fold_a.py", "--download"],
                          capture_output=True, text=True)
    print(proc.stdout[-2000:])
    if not have_corpus():
        print(proc.stderr[-2000:])
        raise SystemExit("corpus build failed; refusing to substitute synthetic prompts")
    if DRIVE_ROOT.exists():
        import shutil
        DRIVE_FOLD.mkdir(parents=True, exist_ok=True)
        for f in FOLD_A.glob("*.jsonl"):
            shutil.copy(f, DRIVE_FOLD / f.name)
        print("[corpus] cached to Drive")

for f in NEEDED:
    n = sum(1 for ln in (FOLD_A / f).read_text(encoding="utf-8").splitlines() if ln.strip())
    print(f"  {f}: {n} prompts")


## 4 — ARMS B and C: does anything replicate across scale and family?

The same RTN ladder, generations, and gold-labelled reasoning, on a 3 B model and
on a different family. Layer is auto-selected at mid-depth from each model's own
config — a fixed index means different things in a 28-layer and a 32-layer model.

**What would be informative:** a reasoning collapse at a similar bits/param on
both. **What would refute generality:** collapse points a bit or more apart, or a
model where accuracy degrades smoothly with no cliff at all.


In [ ]:
for tag, model in MODELS:
    free_vram()
    run_step(f"behavioural-{tag}", "run_behavioural_ladder.py",
             ["--model", model, "--n", str(N_PROMPTS), "--bits", *BITS,
              "--seed", str(SEED), "--label", f"colab-behavioural-{tag}"])
    free_vram()
    run_step(f"gsm8k-{tag}", "run_sector_ladder.py",
             ["--model", model, "--n", str(N_GSM8K), "--bits", *BITS,
              "--label", f"colab-gsm8k-{tag}"])


## 5 — ARM A: the 7 B judge

**The point of this notebook.** Every safety number this project has produced was
made by an instrument that failed:

- a refusal **phrase list** — the reported flip rate moved between 10.4 % and
  48.0 % depending on which strings it contained;
- a 1.5 B **self-judge** — saturated at 100 % REFUSE, including on
  *"I'm sorry to hear you're having trouble finding a present for your brother"*.

A 7 B judge is the smallest instrument with a plausible chance of doing better.
The script reports it **alongside** all four phrase-list variants and prints
every disagreement, so if this judge also fails it will be visible rather than
quietly adopted.

It is still not ground truth, and validation against blinded human labels remains
a prerequisite before any number from it is called a safety rate.


In [ ]:
free_vram()
for tag, _ in MODELS:
    run_dir = latest_run(f"*colab-behavioural-{tag}")
    if run_dir is None:
        print(f"[{tag}] no behavioural run found — skipping judge")
        continue
    args = ["--judge-model", JUDGE_MODEL, str(run_dir)]
    if JUDGE_4BIT:
        args.insert(0, "--judge-4bit")
    run_step(f"judge-{tag}", "classify_completions_judge.py", args)
    free_vram()


## 6 — Estimand and power analyses

Cheap, CPU-only, and run against the saved activations. Both exist because
earlier versions of this work got them wrong:

- **probe transfer** — every d′ was once computed by refitting the direction on
  each scheme's own activations, which answers "can a *fresh* probe be trained
  here", not "does the *original* readout survive". The two diverge sharply.
- **power** — "d′ did not move" is uninformative without a minimum detectable
  effect.


In [ ]:
for tag, _ in MODELS:
    run_dir = latest_run(f"*colab-behavioural-{tag}")
    if run_dir is None:
        continue
    run_step(f"transfer-{tag}", "analyse_probe_transfer.py", [str(run_dir), "--splits", "50"])
    run_step(f"power-{tag}", "analyse_dprime_power.py", [str(run_dir), "--bootstrap", "300"])


## 7 — Export

One zip containing every run directory plus a status summary. Download it, put it
in the repo root, unzip.


In [ ]:
import shutil

runs = sorted(pathlib.Path("artifacts/runs").glob("*colab-*"))
print("run directories produced:")
for r in runs:
    print(f"  {r.name}")
if not runs:
    raise SystemExit("no colab run directories — every arm failed. Read the logs above.")

stamp = time.strftime("%Y%m%d-%H%M%S")
staging = (pathlib.Path("/content") if IN_COLAB else pathlib.Path(".")) / f"cliffguard_colab_{stamp}"
staging.mkdir(parents=True, exist_ok=True)
for r in runs:
    shutil.copytree(r, staging / "artifacts" / "runs" / r.name, dirs_exist_ok=True)

summary = {
    "generated_utc": stamp,
    "gpu": GPU_NAME, "vram_gb": VRAM_GB,
    "torch": torch.__version__, "transformers": transformers.__version__,
    "models_under_test": {t: m for t, m in MODELS},
    "judge_model": JUDGE_MODEL, "judge_4bit": bool(JUDGE_4BIT),
    "n_prompts_per_class": N_PROMPTS, "n_gsm8k": N_GSM8K, "bits": BITS, "seed": SEED,
    "steps": RESULTS,
}
(staging / "colab_status.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

zip_path = pathlib.Path(shutil.make_archive(str(staging), "zip",
                                            root_dir=staging.parent, base_dir=staging.name))
print(f"\n{zip_path}  ({zip_path.stat().st_size/1e6:.1f} MB)")

if DRIVE_ROOT.exists():
    shutil.copy(zip_path, DRIVE_ROOT / zip_path.name)
    print(f"mirrored to {DRIVE_ROOT / zip_path.name}")

print("\nstep status:")
for label, info in RESULTS.items():
    rc = info.get("returncode")
    print(f"  {label:24s} {'OK' if rc == 0 else f'FAILED rc={rc}':12s} {info['minutes']:.1f} min")

failed = [k for k, v in RESULTS.items() if v.get("returncode") != 0]
if failed:
    print(f"\nFAILED STEPS: {failed}")
    print("The zip still contains everything that did complete. Report the failures.")

if IN_COLAB:
    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception as exc:
        print("auto-download failed — take it from the Files pane or Drive:", exc)

print(f"""
DONE.

  1. Download {zip_path.name} (should have started automatically).
  2. Unzip it at the repository root — run directories land in artifacts/runs/.
  3. Say "colab results are in".
""")


---
## Reading the outcome

Decide what each row means **before** looking at the numbers.

| Result | Meaning |
|---|---|
| The 7 B judge produces a non-degenerate split (not ~100 % one class) and broadly agrees with the stricter phrase-list variants | **The project has a safety instrument for the first time.** Everything blocked on that becomes measurable. |
| The 7 B judge also saturates | Two instrument classes have failed. The next step is a purpose-built safety classifier or human labels — not another prompt-based judge. |
| GSM8K collapses near the same bits/param on 3 B and Phi-3.5 | The reasoning cliff is a property of the quantizer, not of one checkpoint. This is the generality result. |
| The collapse points differ by ≥ 1 bit | The cliff is model-dependent; `b*` must be measured per model and no universal threshold exists. |
| Accuracy degrades smoothly with no cliff on some model | The threshold-crossing account is wrong for that model, and that is the most interesting negative outcome available here. |
| Frozen-probe retention stays high where behaviour degrades | The probe is not a safety certificate — consistent with the local finding, now on two more models. |

### Standing limits

- Greedy decoding only. One completion estimates a deterministic decision, not
  behaviour under sampling.
- No independent harmfulness annotation. Labels describe *what the model does*,
  not whether the request was harmful.
- No human validation of any classifier.
- The judge runs quantized (NF4) on a T4. Recorded in the manifest; re-run it in
  fp16 on a larger GPU before publishing anything that depends on it.
